<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet6%20(ANN)/Praktikum%204%20(JS6).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install annoy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520035 sha256=cfc47b6736ee729e66684f6c3d4154217cd60afbf5fd7008a08c0f734fd6aee0
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy


In [5]:
!pip install faiss-cpu
#!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 66.6 MB/s eta 0:00:00


In [7]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679798 sha256=e6cf383744ba64aa2bfaa65e2d4c86146130dfc05e89f41183102350e8dbb723
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [8]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== Annoy ===
Build time: 26.13199257850647 detik
Query time: 0.00043582916259765625 detik
Neighbors: [137425, 115779, 587379, 19925, 453178] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.015911102294921875 detik
Query time: 0.009839534759521484 detik
Neighbors: [137425 115779 587379  19925 454561] ...

=== HNSW (hnswlib) ===
Build time: 169.26336312294006 detik
Query time: 0.000400543212890625 detik
Neighbors: [137425 115779 587379  19925 454561] ...


Lakukan percobaan pada metric distance yang berbeda. catat hasilnya pada tabel yang anda buat sendiri seperti pada praktikum 1.

In [11]:
import pandas as pd

# (Kode pengujian variasi Praktikum 4 kamu di bawahnya...)
df_praktikum4 = pd.DataFrame(results)
display(df_praktikum4)

# Pengujian variasi Metric Distance (Euclidean vs Angular/Cosine)
np.random.seed(42)
metrics = ["Euclidean", "Angular / Cosine"]
results = []

for metric in metrics:
    X_test = X.copy()
    q_test = query.copy()

    # --- 1. ANNOY ---
    ann_metric = 'euclidean' if metric == "Euclidean" else 'angular'
    ann_idx = AnnoyIndex(dim, ann_metric)

    t0 = time.time()
    for i in range(n_data):
        ann_idx.add_item(i, X_test[i])
    ann_idx.build(10)
    b_ann = time.time() - t0

    t0 = time.time()
    res_ann = ann_idx.get_nns_by_vector(q_test[0], k)
    q_ann = time.time() - t0

    results.append({
        "Metric Distance": metric,
        "Model / Algoritma": "Annoy (10 trees)",
        "Build Time (s)": round(b_ann, 4),
        "Query Time (s)": round(q_ann, 6),
        "Top 5 Neighbors": str(res_ann[:5])
    })

    # --- 2. FAISS ---
    if metric == "Euclidean":
        faiss_idx = faiss.IndexFlatL2(dim)
        t0 = time.time()
        faiss_idx.add(X_test)
        b_faiss = time.time() - t0

        t0 = time.time()
        _, res_faiss = faiss_idx.search(q_test, k)
        q_faiss = time.time() - t0
    else:
        # Normalisasi vektor untuk Inner Product / Cosine
        faiss.normalize_L2(X_test)
        faiss.normalize_L2(q_test)
        faiss_idx = faiss.IndexFlatIP(dim)

        t0 = time.time()
        faiss_idx.add(X_test)
        b_faiss = time.time() - t0

        t0 = time.time()
        _, res_faiss = faiss_idx.search(q_test, k)
        q_faiss = time.time() - t0

    results.append({
        "Metric Distance": metric,
        "Model / Algoritma": "FAISS (IndexFlat)",
        "Build Time (s)": round(b_faiss, 4),
        "Query Time (s)": round(q_faiss, 6),
        "Top 5 Neighbors": str(res_faiss[0][:5].tolist())
    })

    # --- 3. HNSW ---
    hnsw_space = 'l2' if metric == "Euclidean" else 'cosine'
    hnsw_idx = hnswlib.Index(space=hnsw_space, dim=dim)

    t0 = time.time()
    hnsw_idx.init_index(max_elements=n_data, ef_construction=200, M=16)
    hnsw_idx.add_items(X_test)
    b_hnsw = time.time() - t0

    hnsw_idx.set_ef(50)
    t0 = time.time()
    res_hnsw, _ = hnsw_idx.knn_query(q_test, k=k)
    q_hnsw = time.time() - t0

    results.append({
        "Metric Distance": metric,
        "Model / Algoritma": "HNSW (hnswlib)",
        "Build Time (s)": round(b_hnsw, 4),
        "Query Time (s)": round(q_hnsw, 6),
        "Top 5 Neighbors": str(res_hnsw[0][:5].tolist())
    })

df_praktikum4 = pd.DataFrame(results)

,Metric Distance,Model / Algoritma,Build Time (s),Query Time (s),Top 5 Neighbors
0,Euclidean,Annoy (10 trees),20.7490,0.000117,"[137425, 115779, 587379, 19925, 453178]"
1,Euclidean,FAISS (IndexFlat),0.0084,0.008469,"[137425, 115779, 587379, 19925, 454561]"
2,Euclidean,HNSW (hnswlib),173.9846,0.000149,"[137425, 115779, 587379, 19925, 454561]"
3,Angular / Cosine,Annoy (10 trees),28.0895,0.000105,"[841933, 44693, 410855, 369214, 137425]"
4,Angular / Cosine,FAISS (IndexFlat),0.0082,0.003317,"[841933, 44693, 410855, 369214, 137425]"
5,Angular / Cosine,HNSW (hnswlib),165.8026,0.000200,"[841933, 44693, 410855, 369214, 137425]"
